In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


# 1. Load Dataset and Check Initial Shape

The customer churn dataset is loaded and its initial dimensions are checked before performing any preprocessing operations.

In [5]:

df = pd.read_csv("../dataset/customer_churn_dataset-testing-master.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [7]:
print("Initial number of records:", df.shape[0])
print("Initial number of features:", df.shape[1])
print("Initial dataset shape:", df.shape)

Initial number of records: 64374
Initial number of features: 12
Initial dataset shape: (64374, 12)


In [8]:
missing_data = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum() / len(df)) * 100
})

missing_data

,Missing Values,Percentage
CustomerID,0,0.0
Age,0,0.0
Gender,0,0.0
Tenure,0,0.0
Usage Frequency,0,0.0
Support Calls,0,0.0
Payment Delay,0,0.0
Subscription Type,0,0.0
Contract Length,0,0.0
Total Spend,0,0.0


In [9]:
total_missing = df.isnull().sum().sum()

print("Total missing values:", total_missing)

if total_missing == 0:
    print("No missing values found in the dataset.")
else:
    print("Missing values are present and require treatment.")

Total missing values: 0
No missing values found in the dataset.


In [10]:
rows_with_missing = df[df.isnull().any(axis=1)]

print("Number of rows containing missing values:", len(rows_with_missing))

rows_with_missing.head(10)

Number of rows containing missing values: 0


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn


## 2. Duplicate Analysis and Removal

Duplicate records are identified and removed to avoid repeated observations from affecting the machine learning models.

In [11]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 0


In [12]:
df = df.drop_duplicates()

print("Duplicate records removed successfully.")

Duplicate records removed successfully.


In [13]:
print("Dataset shape after removing duplicates:", df.shape)

Dataset shape after removing duplicates: (64374, 12)


## 3. Separation of Features and Target

The target variable `Churn` is separated from the input features. The input features are stored in `X`, while the target variable is stored in `y`.

In [14]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

Features (X) shape: (64374, 11)
Target (y) shape: (64374,)


In [15]:
print("Input Features (X):")
display(X.head())

print("\nTarget Variable (y):")
display(y.head())

Input Features (X):


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20
2,3,47,Male,27,10,2,29,Premium,Annual,757,21
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18
4,5,53,Female,58,24,9,2,Standard,Annual,533,18



Target Variable (y):


0    1
1    0
2    0
3    0
4    0
Name: Churn, dtype: int64

## 4. Train-Test Split

The dataset is divided into training and testing subsets. The training set is used for model learning, while the testing set is reserved for evaluating model performance on unseen data. Stratified splitting is used to maintain the class distribution of the target variable.

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training features shape: (51499, 11)
Testing features shape: (12875, 11)
Training target shape: (51499,)
Testing target shape: (12875,)


## 5. Identification of Numerical and Categorical Features

The input features are separated into numerical and categorical features. Numerical features will be standardized, while categorical features will be converted into numerical form using one-hot encoding.

In [20]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['CustomerID', 'Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction']

Categorical features:
['Gender', 'Subscription Type', 'Contract Length']


In [21]:
print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Number of numerical features: 8
Number of categorical features: 3


## 6. Encoding and Feature Scaling

Numerical features are standardized using StandardScaler, while categorical features are transformed into numerical form using OneHotEncoder. A ColumnTransformer is used to apply the appropriate preprocessing operation to each group of features.

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


## 7. Applying Preprocessing

The preprocessing pipeline is fitted only on the training data and then applied to both the training and testing datasets. This prevents information from the testing dataset from influencing the preprocessing process and helps avoid data leakage.

In [23]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training data preprocessing completed.")
print("Testing data preprocessing completed.")

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape:", X_test_processed.shape)


Training data preprocessing completed.
Testing data preprocessing completed.
Training processed shape: (51499, 16)
Testing processed shape: (12875, 16)


In [24]:
print("X_train_processed exists:", "X_train_processed" in globals())
print("X_test_processed exists:", "X_test_processed" in globals())

X_train_processed exists: True
X_test_processed exists: True


In [25]:
print(type(X_train_processed))
print(X_train_processed.shape)

<class 'numpy.ndarray'>
(51499, 16)


In [27]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("\nOriginal testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (51499, 11)
Processed training shape: (51499, 16)

Original testing shape: (12875, 11)
Processed testing shape: (12875, 16)


In [28]:
print("Processed training data type:", type(X_train_processed))
print("Processed testing data type:", type(X_test_processed))

print("\nProcessed training data shape:", X_train_processed.shape)
print("Processed testing data shape:", X_test_processed.shape)

Processed training data type: <class 'numpy.ndarray'>
Processed testing data type: <class 'numpy.ndarray'>

Processed training data shape: (51499, 16)
Processed testing data shape: (12875, 16)


In [29]:
print("Missing values in processed training data:",
      np.isnan(X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed).sum())

print("Missing values in processed testing data:",
      np.isnan(X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed).sum())

Missing values in processed training data: 0
Missing values in processed testing data: 0


## 8. Final Preprocessing Summary

The dataset was prepared for machine learning by performing duplicate removal, separating the input features and target variable, splitting the data into training and testing sets, one-hot encoding categorical features, and standardizing numerical features.

The preprocessing transformations were fitted only on the training data and subsequently applied to the testing data to prevent data leakage.

In [30]:
print("Initial dataset shape:", df.shape)

print("\nTraining data shape before preprocessing:", X_train.shape)
print("Testing data shape before preprocessing:", X_test.shape)

print("\nTraining data shape after preprocessing:", X_train_processed.shape)
print("Testing data shape after preprocessing:", X_test_processed.shape)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

print("\nPreprocessing completed successfully!")

Initial dataset shape: (64374, 12)

Training data shape before preprocessing: (51499, 11)
Testing data shape before preprocessing: (12875, 11)

Training data shape after preprocessing: (51499, 16)
Testing data shape after preprocessing: (12875, 16)

Number of numerical features: 8
Number of categorical features: 3

Preprocessing completed successfully!


## 9. Data Leakage Verification

To prevent data leakage, the preprocessing pipeline was fitted only on the training data. The same fitted transformations were then applied to the testing data without fitting again on the test set.

In [3]:
import joblib
from pathlib import Path

# Create folder for preprocessed data
artifact_dir = Path("../model/preprocessed")
artifact_dir.mkdir(parents=True, exist_ok=True)

# Save processed training and testing data
joblib.dump(X_train_processed, artifact_dir / "X_train_processed.pkl")
joblib.dump(X_test_processed, artifact_dir / "X_test_processed.pkl")

# Save target variables
joblib.dump(y_train, artifact_dir / "y_train.pkl")
joblib.dump(y_test, artifact_dir / "y_test.pkl")

# Save feature names
feature_names = preprocessor.get_feature_names_out()
joblib.dump(feature_names, artifact_dir / "feature_names.pkl")

print("Preprocessed data saved successfully!")
print("Number of processed features:", len(feature_names))
print("Saved location:", artifact_dir)

NameError: name 'X_train_processed' is not defined